# Article extraction and location matching

This notebook converts the Nexis/NU.nl DOCX files into CSV files that can be used for news value classification and GIS analysis.

The notebook creates two main outputs:

1. `articles_for_classification.csv`  
   One row per article, including title, date, subjects, body text, lead text, and extracted Dutch locations.

2. `location_mentions_for_gis.csv`  
   One row per article-location combination, including coordinates. This file is later combined with the news value scores for the GIS analysis.

I also save `articles_for_classification_with_locations_only.csv`, because only articles with at least one Dutch location are used for the location-based news value analysis.

In [ ]:
# Connect Google Drive
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Install packages


In [ ]:
# These packages are needed for reading DOCX files, working with tables,
# and recognising Dutch place names.
!pip install -q python-docx pandas spacy
!python -m spacy download nl_core_news_lg

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 9.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 568.1/568.1 MB 897.3 kB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('nl_core_news_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import json
import os
import re
import time
from datetime import datetime
from pathlib import Path

import docx
import pandas as pd
import spacy
from pandas.errors import EmptyDataError

## 3. Set project folders

Here I define the folders used in the project. The DOCX files are stored in year/month folders inside the `data` folder. The processed CSV files are saved in `data_processed`.


In [ ]:
PROJECT_DIR = Path("/content/drive/MyDrive/Thesis")
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "data_processed"
BATCH_DIR = OUTPUT_DIR / "batches"

OSM_JSON_PATH = PROJECT_DIR / "osm.json"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BATCH_DIR.mkdir(parents=True, exist_ok=True)

ARTICLE_DIRS = sorted([
    folder for folder in DATA_DIR.iterdir()
    if folder.is_dir() and folder.name.endswith("-2025")
])

print("Project folder:", PROJECT_DIR)
print("Data folder:", DATA_DIR)
print("Output folder:", OUTPUT_DIR)
print("Batch folder:", BATCH_DIR)

print("\nFound article folders:")
for folder in ARTICLE_DIRS:
    print(folder)

print("\nOSM file exists:", OSM_JSON_PATH.exists(), OSM_JSON_PATH)

Project folder: /content/drive/MyDrive/Thesis
Data folder: /content/drive/MyDrive/Thesis/data
Output folder: /content/drive/MyDrive/Thesis/data_processed
Batch folder: /content/drive/MyDrive/Thesis/data_processed/batches

Found article folders:
/content/drive/MyDrive/Thesis/data/01-2025
/content/drive/MyDrive/Thesis/data/02-2025
/content/drive/MyDrive/Thesis/data/03-2025
/content/drive/MyDrive/Thesis/data/04-2025
/content/drive/MyDrive/Thesis/data/05-2025
/content/drive/MyDrive/Thesis/data/06-2025
/content/drive/MyDrive/Thesis/data/07-2025
/content/drive/MyDrive/Thesis/data/08-2025
/content/drive/MyDrive/Thesis/data/09-2025
/content/drive/MyDrive/Thesis/data/10-2025
/content/drive/MyDrive/Thesis/data/11-2025
/content/drive/MyDrive/Thesis/data/12-2025

OSM file exists: True /content/drive/MyDrive/Thesis/osm.json


## 4. Load the Dutch NLP model and Dutch place names

I use spaCy to detect place names in the article text. I then keep only place names that also occur in the Dutch OSM location file, so that non-Dutch or ambiguous locations are filtered out.

In [ ]:
# Load Dutch NLP model
nlp = spacy.load("nl_core_news_lg")

# Load OSM JSON
with open(OSM_JSON_PATH, "r", encoding="utf-8") as osm_file:
    osm_json = json.load(osm_file)

osm_df = pd.json_normalize(osm_json["elements"], sep=".")

# The coordinates can be stored in slightly different columns, depending on the OSM export.
name_col = "tags.name"
if "lat" in osm_df.columns and "lon" in osm_df.columns:
    lat_col, lon_col = "lat", "lon"
elif "center.lat" in osm_df.columns and "center.lon" in osm_df.columns:
    lat_col, lon_col = "center.lat", "center.lon"
else:
    raise ValueError("Could not find latitude/longitude columns in osm.json.")

osm_locations = (
    osm_df[[name_col, lat_col, lon_col]]
    .dropna()
    .rename(columns={name_col: "location", lat_col: "lat", lon_col: "lon"})
)

# If OSM has duplicate names, keep the first coordinate.
duplicate_location_count = osm_locations["location"].duplicated().sum()
print("Duplicate OSM location names:", duplicate_location_count)

osm_locations = osm_locations.drop_duplicates(subset=["location"], keep="first").copy()
osm_lookup = osm_locations.set_index("location")[["lat", "lon"]]
valid_dutch_locations = set(osm_lookup.index)

print("Number of Dutch locations loaded:", len(valid_dutch_locations))
osm_locations.head()

Duplicate OSM location names: 60
Number of Dutch locations loaded: 2435


,location,lat,lon
0,Nuenen,51.474302,5.549044
1,Geldrop,51.422668,5.560755
2,Assen,52.995227,6.560498
3,Ubbena,53.052397,6.574640
4,Taarlo,53.032981,6.626097


## 5. Functions for reading articles and finding locations

The functions below read the Nexis DOCX files, extract the main article information, and find Dutch locations in the article body. I also replace some football club names before location extraction, because these names sometimes caused false location matches.


In [ ]:
# Some football club names include city names.
# I replace these names before location extraction to avoid counting the city as a news location.

FOOTBALL_REPLACEMENTS = {
    # Eredivisie
    "PSV Eindhoven": "PSV",
    "Feyenoord Rotterdam": "Feyenoord",
    "Ajax Amsterdam": "Ajax",
    "NEC Nijmegen": "NEC",
    "FC Groningen": "FCG",
    "AZ Alkmaar": "AZ",
    "FC Twente": "FCT",
    "FC Utrecht": "FCU",
    "SC Heerenveen": "SCH",
    "sc Heerenveen": "SCH",
    "Sc Heerenveen": "SCH",
    "Sparta Rotterdam": "Sparta",
    "Fortuna Sittard": "Fortuna",
    "Excelsior Rotterdam": "Excelsior",
    "Go Ahead Eagles Deventer": "Go Ahead Eagles",
    "PEC Zwolle": "PEC",
    "FC Volendam": "FCV",
    "Heracles Almelo": "Heracles",
    "NAC Breda": "NAC",
    # Eerste Divisie
    "ADO Den Haag": "ADO",
    "Cambuur Leeuwarden": "Cambuur",
    "De Graafschap Doetinchem": "De Graafschap",
    "Roda JC Kerkrade": "Roda JC",
    "Almere City": "ACFC",
    "VVV Venlo": "VVV",
    "Willem II Tilburg": "Willem II",
    "RKC Waalwijk": "RKC",
    "FC Emmen": "FCE",
    "FC Eindhoven": "FCE",
    "FC Dordrecht": "FCD",
    "Helmond Sport": "HSport",
    "MVV Maastricht": "MVV",
    "TOP Oss": "TOPO",
    "Vitesse Arnhem": "Vitesse",
}


def clean_text_for_location_extraction(text):
    for old_name, new_name in FOOTBALL_REPLACEMENTS.items():
        text = text.replace(old_name, new_name)
    return text


def parse_subjects(text):
    # Extract the Nexis subject labels from the Subject paragraph.
    matches = re.findall(r"\s([^;:]+)\s\((\d+)%\)", text)
    subjects = [{"subject": subject.strip(), "percentage": int(percentage)} for subject, percentage in matches]
    subjects_text = "; ".join([s["subject"] for s in subjects])
    subjects_json = json.dumps(subjects, ensure_ascii=False)
    return subjects_text, subjects_json


def parse_docx_article(article_path):
    # Read one Nexis DOCX article and return the information I need later.
    # The title and date are usually in fixed paragraph positions in the export.
    article = docx.Document(str(article_path))
    paragraphs = [p.text.strip() for p in article.paragraphs]
    non_empty_paragraphs = [p for p in paragraphs if p]

    # Use index-based logic, with safe fallbacks.
    title = paragraphs[2].strip() if len(paragraphs) > 2 else (non_empty_paragraphs[0] if non_empty_paragraphs else "")
    date = paragraphs[4].strip() if len(paragraphs) > 4 else ""

    subjects_text = ""
    subjects_json = "[]"

    body_paragraph = False
    body_paragraphs = []

    for paragraph_text in paragraphs:
        text = paragraph_text.strip()

        if text.startswith("Subject:"):
            subjects_text, subjects_json = parse_subjects(text)

        elif text == "Body":
            body_paragraph = True
            continue

        elif text == "Classification":
            body_paragraph = False
            continue

        elif text == "PDF-bestand van dit document":
            continue

        elif body_paragraph and text:
            body_paragraphs.append(text)

    body_text = "\n".join(body_paragraphs)
    lead_text = " ".join(body_paragraphs[:3])

    return {
        "article_id": article_path.stem,
        "file": str(article_path),
        "file_name": article_path.name,
        "title": title,
        "date": date,
        "subjects": subjects_text,
        "subjects_json": subjects_json,
        "body_text": body_text,
        "lead_text": lead_text,
        "body_char_count": len(body_text),
        "lead_char_count": len(lead_text),
    }


def extract_dutch_locations(body_text):
    cleaned_text = clean_text_for_location_extraction(body_text)

    if len(cleaned_text) > nlp.max_length:
        cleaned_text = cleaned_text[:nlp.max_length]

    doc = nlp(cleaned_text)

    locations = []
    for ent in doc.ents:
        place_name = ent.text.strip()

        if ent.label_ == "GPE" and place_name in valid_dutch_locations:
            locations.append(place_name)

    return sorted(set(locations))


## 6. Find the DOCX article files

In this step I collect all DOCX files from the monthly folders.

In [ ]:
article_paths = []

for article_dir in ARTICLE_DIRS:
    article_paths.extend(sorted(article_dir.glob("*.DOCX")))
    article_paths.extend(sorted(article_dir.glob("*.docx")))

article_paths = sorted(set(article_paths))

print("Number of DOCX files found:", len(article_paths))
if len(article_paths) == 0:
    print("Check ARTICLE_DIRS. No DOCX files were found.")

Number of DOCX files found: 55772


## 7. Extract article information and locations in batches

The full article dataset is quite large, so I process the DOCX files in batches. Google Colab crashed during some earlier attempts, so I used ChatGPT as coding support to help me implement intermediate batch saving and a resume-safe workflow. This allows the notebook to skip batches that were already saved and continue without starting over.

In [ ]:
LOCATION_COLUMNS = [
    "article_id", "file", "file_name", "title", "date",
    "subjects", "location", "lat", "lon"
]

BATCH_SIZE = 500

total_articles = len(article_paths)

start_time = time.time()
last_timestamp = start_time


def save_batch(df, path):
    # Save one batch file.
    df.to_csv(path, index=False, encoding="utf-8-sig")


for batch_start in range(0, total_articles, BATCH_SIZE):
    batch_end = min(batch_start + BATCH_SIZE, total_articles)

    articles_batch_file = BATCH_DIR / f"articles_{batch_start + 1:06d}_{batch_end:06d}.csv"
    locations_batch_file = BATCH_DIR / f"locations_{batch_start + 1:06d}_{batch_end:06d}.csv"
    errors_batch_file = BATCH_DIR / f"errors_{batch_start + 1:06d}_{batch_end:06d}.csv"

    # If both files already exist, skip this batch when restarting Colab
    if articles_batch_file.exists() and locations_batch_file.exists():
        print(f"Skipping batch {batch_start + 1}-{batch_end}, already saved.")
        continue

    print(f"\nStarting batch {batch_start + 1}-{batch_end} at {datetime.now()}", flush=True)

    article_rows = []
    location_rows = []
    error_rows = []

    batch_time = time.time()

    for local_i, article_path in enumerate(article_paths[batch_start:batch_end], start=1):
        global_i = batch_start + local_i
        now = time.time()

        # Print progress every 10 minutes
        if now - last_timestamp >= 600:
            elapsed_minutes = (now - start_time) / 60
            print(
                f"{datetime.now()} | Processed {global_i} / {total_articles} articles | "
                f"Elapsed: {elapsed_minutes:.1f} minutes",
                flush=True
            )
            last_timestamp = now

        # Also print every 500 articles
        if global_i % 500 == 0:
            print(f"Processed {global_i} / {total_articles} articles", flush=True)

        try:
            article_data = parse_docx_article(article_path)
            locations = extract_dutch_locations(article_data["body_text"])

            locations_with_coordinates = []

            for loc in locations:
                # Safer than crashing if one location is missing from osm_lookup
                if loc not in osm_lookup.index:
                    continue

                lat = osm_lookup.loc[loc, "lat"]
                lon = osm_lookup.loc[loc, "lon"]

                # If osm_lookup has duplicate index values, loc may return a Series.
                # In that case, take the first match.
                if hasattr(lat, "iloc"):
                    lat = lat.iloc[0]
                if hasattr(lon, "iloc"):
                    lon = lon.iloc[0]

                locations_with_coordinates.append({
                    "location": loc,
                    "lat": float(lat),
                    "lon": float(lon)
                })

                location_rows.append({
                    "article_id": article_data["article_id"],
                    "file": article_data["file"],
                    "file_name": article_data["file_name"],
                    "title": article_data["title"],
                    "date": article_data["date"],
                    "subjects": article_data["subjects"],
                    "location": loc,
                    "lat": float(lat),
                    "lon": float(lon),
                })

            article_rows.append({
                **article_data,
                "locations": "; ".join(locations),
                "locations_json": json.dumps(locations_with_coordinates, ensure_ascii=False),
                "location_count": len(locations),
                "has_dutch_location": len(locations) > 0,
                "entertainment_score": "",
                "bad_news_score": "",
                "magnitude_score": "",
                "good_news_score": "",
                "celebrity_score": "",
                "power_elite_score": "",
            })

        except Exception as e:
            error_rows.append({
                "global_i": global_i,
                "article_path": str(article_path),
                "error": repr(e)
            })

    # Save this batch
    save_batch(pd.DataFrame(article_rows), articles_batch_file)
    save_batch(pd.DataFrame(location_rows, columns=LOCATION_COLUMNS), locations_batch_file)

    if error_rows:
        save_batch(pd.DataFrame(error_rows), errors_batch_file)

    batch_minutes = (time.time() - batch_time) / 60
    print(
        f"{datetime.now()} | Saved batch {batch_start + 1}-{batch_end} | "
        f"Articles: {len(article_rows)} | "
        f"Locations: {len(location_rows)} | "
        f"Errors: {len(error_rows)} | "
        f"Batch time: {batch_minutes:.1f} min",
        flush=True
    )

## 8. Check saved batch files

In [ ]:
# Check whether any location batch files are empty or unreadable.
location_batch_files = sorted(BATCH_DIR.glob("locations_*.csv"))

print("Number of location files:", len(location_batch_files))

problem_files = []

for file in location_batch_files:
    try:
        df_test = pd.read_csv(file, nrows=5)
    except EmptyDataError:
        problem_files.append((file.name, file.stat().st_size))
    except Exception as e:
        problem_files.append((file.name, file.stat().st_size, type(e).__name__, str(e)))

print("Problem files:")
for item in problem_files:
    print(item)

Number of location files: 112
Problem files:
('locations_037001_037500.csv', 1)


## 9. Combine the saved batches

The extraction step saves separate batch files. In this step I combine those batches into one article-level file and one location-level file. This can be run again without reprocessing the original DOCX files.

In [ ]:
article_batch_files = sorted(BATCH_DIR.glob("articles_*.csv"))
location_batch_files = sorted(BATCH_DIR.glob("locations_*.csv"))
error_batch_files = sorted(BATCH_DIR.glob("errors_*.csv"))

def read_csv_batches(files, label, columns=None):
    dfs = []
    skipped = []

    for file in files:
        try:
            df = pd.read_csv(file)

            if not df.empty:
                dfs.append(df)

        except EmptyDataError:
            skipped.append(file.name)

    print(label, "files found:", len(files))
    print(label, "non-empty files read:", len(dfs))
    print(label, "empty files:", len(skipped))

    if skipped:
        print("Empty files:")
        for name in skipped:
            print("-", name)

    if not dfs:
        if columns is not None:
            return pd.DataFrame(columns=columns)
        else:
            raise ValueError("No readable " + label + " files found.")

    return pd.concat(dfs, ignore_index=True)

LOCATION_COLUMNS = [
    "article_id", "file", "file_name", "title", "date",
    "subjects", "location", "lat", "lon"
]

articles_df = read_csv_batches(article_batch_files, "article")
locations_df = read_csv_batches(location_batch_files, "location", columns=LOCATION_COLUMNS)

articles_df["locations"] = articles_df["locations"].fillna("")
articles_df["locations_json"] = articles_df["locations_json"].fillna("[]")

articles_df.to_csv(OUTPUT_DIR / "articles_final.csv", index=False)
locations_df.to_csv(OUTPUT_DIR / "locations_final.csv", index=False)

article files found: 112
article non-empty files read: 112
article empty files: 0
location files found: 112
location non-empty files read: 111
location empty files: 1
Empty files:
- locations_037001_037500.csv


In [ ]:
# Check whether the combined data has the expected size.
print("Expected articles:", len(article_paths))
print("Combined articles:", len(articles_df))
print("Combined location mentions:", len(locations_df))

if len(articles_df) != len(article_paths):
    print("Warning: the number of combined articles does not match the number of DOCX files.")
else:
    print("Article count matches.")

Expected articles: 55772
Combined articles: 55772
Combined location mentions: 19802
Article count matches.


## 10. Save the final CSV files

Here I save the final files used in the next steps of the thesis. The file with only articles that mention Dutch locations is the main input for the news value classification, because the GIS analysis is based on place mentions.


In [ ]:
articles_csv = OUTPUT_DIR / "articles_for_classification.csv"
articles_with_locations_csv = OUTPUT_DIR / "articles_for_classification_with_locations_only.csv"
locations_csv = OUTPUT_DIR / "location_mentions_for_gis.csv"

articles_df.to_csv(articles_csv, index=False, encoding="utf-8-sig")
articles_df[articles_df["has_dutch_location"]].to_csv(articles_with_locations_csv, index=False, encoding="utf-8-sig")
locations_df.to_csv(locations_csv, index=False, encoding="utf-8-sig")

print("Saved:")
print("-", articles_csv)
print("-", articles_with_locations_csv)
print("-", locations_csv)

Saved:
- /content/drive/MyDrive/Thesis/data_processed/articles_for_classification.csv
- /content/drive/MyDrive/Thesis/data_processed/articles_for_classification_with_locations_only.csv
- /content/drive/MyDrive/Thesis/data_processed/location_mentions_for_gis.csv


## 11. Quick inspection of the output

In [ ]:
print("Articles with at least one Dutch location:", articles_df["has_dutch_location"].sum())
print("Average number of Dutch locations per article:", articles_df["location_count"].mean())

articles_df[["article_id", "title", "date", "location_count", "locations"]].head()

Articles with at least one Dutch location: 12244
Average number of Dutch locations per article: 0.3550527146238256


,article_id,title,date,location_count,locations
0,'Vrienden' Poetin en Xi verstevigen band in vi...,'Vrienden' Poetin en Xi verstevigen band in vi...,21 januari 2025 dinsdag 10:11 PM GMT,0,
1,10 procent van huishoudens bezit ruim de helft...,10 procent van huishoudens bezit ruim de helft...,14 januari 2025 dinsdag 03:41 PM GMT,0,
2,10 procent van huishoudens bezit ruim helft va...,10 procent van huishoudens bezit ruim helft va...,14 januari 2025 dinsdag 03:41 PM GMT,0,
3,"100 gewonden bij Zaporizhzhia, VS schenkt 500 ...","100 gewonden bij Zaporizhzhia, VS schenkt 500 ...",9 januari 2025 donderdag 12:45 PM GMT,0,
4,"100 gewonden bij Zaporizhzhia, VS schenkt 500 ...","100 gewonden bij Zaporizhzhia, VS schenkt 500 ...",9 januari 2025 donderdag 12:45 PM GMT,0,
